# SPG Dataset — End-to-End Benchmark Notebook
Single-run pipeline: load `spg.csv`, preprocess, train baselines + probabilistic transformer,
calibrate intervals, compute metrics, save CSVs & figures to `results_publication/`.


# EMHIRES_WIND_COUNTRY_June2019.xlsx — Benchmark pipeline
Load the Excel, preprocess, create windows, train baselines + probabilistic transformer, calibrate, evaluate and save outputs to `results_publication/`.


In [1]:
# Cell 2
import os, sys, random, json, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")
warnings.filterwarnings("ignore")

# ML libs
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.arima.model import ARIMA
import joblib

# TensorFlow / Keras
import tensorflow as tf

# optional CRPS
try:
    from properscoring import crps_gaussian
    HAS_PROPERSCORING = True
except Exception:
    HAS_PROPERSCORING = False

# Config - tune for final runs
LOOKBACK = 48
HORIZON = 24
SEED = 42
RF_TREES = 300
GB_TREES = 300
LSTM_EPOCHS = 30
TRANSFORMER_EPOCHS = 40
BATCH_SIZE = 64
QUANTILES = [0.1, 0.5, 0.9]

OUTDIR = Path("results_publication")
OUTDIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    try:
        tf.random.set_seed(seed)
    except Exception:
        pass

set_seed(SEED)
print("Ready. CRPS available:", HAS_PROPERSCORING)


Ready. CRPS available: False


In [2]:
# Cell 3: choose filename (just the filename) and preview sheet names
FILENAME = "EMHIRES_WIND_COUNTRY_June2019.xlsx"   # <--- leave this as is if that's your file

SEARCH_DIRS = [Path("data"), Path("datasets"), Path("/mnt/data"), Path(".")]

def find_file_by_name(fname):
    for d in SEARCH_DIRS:
        p = d / fname
        if p.exists():
            return p
    for d in SEARCH_DIRS:
        if d.exists():
            for f in d.iterdir():
                if f.is_file() and f.name.lower() == fname.lower():
                    return f
    raise FileNotFoundError(f"File '{fname}' not found in {SEARCH_DIRS}; put it in data/ or current folder.")

file_path = find_file_by_name(FILENAME)
print("Found file:", file_path)

# Preview sheet names and first rows (will load only sheet0 header for preview)
xls = pd.ExcelFile(file_path)
print("Sheets:", xls.sheet_names)
preview = pd.read_excel(xls, sheet_name=0, nrows=5)
print("Preview of sheet 0 columns:", preview.columns.tolist())
display(preview)


Found file: data\EMHIRES_WIND_COUNTRY_June2019.xlsx
Sheets: ['TS.CF.COUNTRY.30yr.date']
Preview of sheet 0 columns: ['Time_step', 'Date', 'Year', 'Month', 'Day', 'Hour', 'AL', 'AT', 'BE', 'BG', 'BA', 'CH', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI', 'FR', 'HR', 'HU', 'XK', 'IE', 'IS', 'IT', 'LT', 'LU', 'LV', 'ME', 'MK', 'NL', 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK', 'UK']


,Time_step,Date,Year,Month,Day,Hour,AL,AT,BE,BG,...,NL,NO,PL,PT,RO,RS,SE,SI,SK,UK
0,1,1986-01-01,1986,1,1,0,0.3685,0.06084,0.25200,0.23762,...,0.40628,0.26292,0.12015,0.22683,0.05124,0.25050,0.17086,0.67720,0.05715,0.268794
1,2,1986-01-01,1986,1,1,1,0.3235,0.05282,0.23841,0.23842,...,0.39115,0.26376,0.11985,0.25821,0.04665,0.24625,0.15909,0.66776,0.05218,0.270761
2,3,1986-01-01,1986,1,1,2,0.2735,0.04276,0.22110,0.23033,...,0.36163,0.26695,0.12113,0.27921,0.04543,0.23250,0.15296,0.65904,0.03905,0.265209
3,4,1986-01-01,1986,1,1,3,0.2600,0.02914,0.21491,0.24158,...,0.35170,0.27101,0.11858,0.33106,0.04455,0.20525,0.15328,0.58460,0.02130,0.257709
4,5,1986-01-01,1986,1,1,4,0.2530,0.01780,0.20304,0.26168,...,0.36168,0.28097,0.11101,0.38668,0.05438,0.20075,0.15013,0.46732,0.01029,0.247726


In [4]:
# Cell 4 (fixed): load the main sheet (change sheet_name if your data is on a different sheet)
SHEET = 0   # or set to sheet name like "Data" if needed

df_raw = pd.read_excel(file_path, sheet_name=SHEET, engine='openpyxl')
print("Loaded raw shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())

# detect timestamp-like column heuristically
def first_datetime_candidate(df):
    commons = ['timestamp','time','date','datetime','ts','datetime_utc','index']
    cols = list(df.columns)
    # direct matches (case-insensitive)
    for name in commons:
        for c in cols:
            if c.lower() == name:
                return c
    # try to parse each column to datetime and pick one that parses for >70% entries
    for c in cols:
        try:
            parsed = pd.to_datetime(df[c], errors='coerce', dayfirst=False)
            if parsed.notna().mean() > 0.7:
                return c
        except Exception:
            continue
    return None

ts_col = first_datetime_candidate(df_raw)
if ts_col is None:
    print("Could not auto-detect timestamp column. Columns available:\n", df_raw.columns.tolist())
    raise ValueError("Please inspect the column names above and set `ts_col` manually to the timestamp column.")

print("Detected timestamp column:", ts_col)
df_raw[ts_col] = pd.to_datetime(df_raw[ts_col], errors='coerce')
if df_raw[ts_col].isna().all():
    raise ValueError(f"Timestamp column '{ts_col}' could not be parsed to datetimes.")
df_raw = df_raw.set_index(ts_col).sort_index()
print("Index range:", df_raw.index.min(), "->", df_raw.index.max())
display(df_raw.head())


Loaded raw shape: (262968, 42)
Columns: ['Time_step', 'Date', 'Year', 'Month', 'Day', 'Hour', 'AL', 'AT', 'BE', 'BG', 'BA', 'CH', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI', 'FR', 'HR', 'HU', 'XK', 'IE', 'IS', 'IT', 'LT', 'LU', 'LV', 'ME', 'MK', 'NL', 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK', 'UK']
Detected timestamp column: Date
Index range: 1986-01-01 00:00:00 -> 2015-12-31 00:00:00


,Time_step,Year,Month,Day,Hour,AL,AT,BE,BG,BA,...,NL,NO,PL,PT,RO,RS,SE,SI,SK,UK
Date,,,,,,,,,,,,,,,,,,,,,
1986-01-01,1,1986,1,1,0,0.3685,0.06084,0.25200,0.23762,0.2480,...,0.40628,0.26292,0.12015,0.22683,0.05124,0.25050,0.17086,0.67720,0.05715,0.268794
1986-01-01,2,1986,1,1,1,0.3235,0.05282,0.23841,0.23842,0.2270,...,0.39115,0.26376,0.11985,0.25821,0.04665,0.24625,0.15909,0.66776,0.05218,0.270761
1986-01-01,3,1986,1,1,2,0.2735,0.04276,0.22110,0.23033,0.2000,...,0.36163,0.26695,0.12113,0.27921,0.04543,0.23250,0.15296,0.65904,0.03905,0.265209
1986-01-01,4,1986,1,1,3,0.2600,0.02914,0.21491,0.24158,0.1875,...,0.35170,0.27101,0.11858,0.33106,0.04455,0.20525,0.15328,0.58460,0.02130,0.257709
1986-01-01,5,1986,1,1,4,0.2530,0.01780,0.20304,0.26168,0.1690,...,0.36168,0.28097,0.11101,0.38668,0.05438,0.20075,0.15013,0.46732,0.01029,0.247726


In [5]:
# Cell 5: auto-map relevant columns (wind generation, wind speed, demand/load)
print("Columns before mapping:", df_raw.columns.tolist())

def map_relevant_columns(cols):
    mapping = {}
    for c in cols:
        lc = c.lower()
        if 'wind' in lc and ('gen' in lc or 'generation' in lc or 'power' in lc):
            mapping[c] = 'wind_gen'
        elif 'wind' in lc and ('speed' in lc or 'ms' in lc):
            mapping[c] = 'wind_speed'
        elif 'demand' in lc or 'load' in lc or 'consum' in lc:
            mapping[c] = 'demand'
        elif 'solar' in lc or 'pv' in lc or 'ghi' in lc:
            mapping[c] = 'solar'
        # fallback: if numeric and no other match, leave for inspection
    return mapping

colmap = map_relevant_columns(list(df_raw.columns))
print("Auto column mapping choices (rename these):", colmap)
# apply mapping
df_work = df_raw.rename(columns=colmap)
print("Columns after mapping (keep an eye):", df_work.columns.tolist())
display(df_work[list(df_work.columns)[:8]].head())


Columns before mapping: ['Time_step', 'Year', 'Month', 'Day', 'Hour', 'AL', 'AT', 'BE', 'BG', 'BA', 'CH', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI', 'FR', 'HR', 'HU', 'XK', 'IE', 'IS', 'IT', 'LT', 'LU', 'LV', 'ME', 'MK', 'NL', 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK', 'UK']
Auto column mapping choices (rename these): {}
Columns after mapping (keep an eye): ['Time_step', 'Year', 'Month', 'Day', 'Hour', 'AL', 'AT', 'BE', 'BG', 'BA', 'CH', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI', 'FR', 'HR', 'HU', 'XK', 'IE', 'IS', 'IT', 'LT', 'LU', 'LV', 'ME', 'MK', 'NL', 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK', 'UK']


,Time_step,Year,Month,Day,Hour,AL,AT,BE
Date,,,,,,,,
1986-01-01,1,1986,1,1,0,0.3685,0.06084,0.25200
1986-01-01,2,1986,1,1,1,0.3235,0.05282,0.23841
1986-01-01,3,1986,1,1,2,0.2735,0.04276,0.22110
1986-01-01,4,1986,1,1,3,0.2600,0.02914,0.21491
1986-01-01,5,1986,1,1,4,0.2530,0.01780,0.20304


In [9]:
# REPLACE previous window-building cell with this robust version
# It auto-detects available features, fills missing ones with zeros,
# builds windows and splits, and exposes FEATURE_COLUMNS and gen_proxy index.

import numpy as np, pandas as pd
from collections import OrderedDict

# Use global LOOKBACK/HORIZON if set, otherwise default
LOOKBACK = globals().get('LOOKBACK', 48)
HORIZON  = globals().get('HORIZON', 24)

# Detect candidate features in df (df should be preprocessed from earlier cell)
available = list(df.columns)
# Priority order we want: demand (target), solar (generation proxy), wind_gen, wind_speed, hour, dayofyear, weekday
preferred = ['demand','solar','wind_gen','wind_speed','hour','dayofyear','weekday']

# Build final feature list: ensure demand is first feature in Y but we keep it in X also
FEATURE_COLUMNS = []
for p in preferred:
    if p in available and p not in FEATURE_COLUMNS:
        FEATURE_COLUMNS.append(p)

# If demand missing (shouldn't happen because we created fallback), ensure it's present
if 'demand' not in FEATURE_COLUMNS:
    FEATURE_COLUMNS.insert(0, 'demand')  # put demand first and create if missing
    df['demand'] = df.get('demand', 0.0)

# Add any other numeric columns (avoid duplicates)
for c in available:
    if c not in FEATURE_COLUMNS:
        # include only numeric-like columns to avoid text columns
        try:
            tmp = pd.to_numeric(df[c], errors='coerce')
            if tmp.notna().sum() > 0:
                FEATURE_COLUMNS.append(c)
        except Exception:
            continue

print("Feature columns used (in order):", FEATURE_COLUMNS)

# Create a numeric array having exactly these columns (fill missing with zeros)
def build_feature_array(df_local, feat_cols):
    data_cols = []
    for c in feat_cols:
        if c in df_local.columns:
            series = pd.to_numeric(df_local[c], errors='coerce').fillna(0.0)
        else:
            series = pd.Series(0.0, index=df_local.index)
        data_cols.append(series.values.reshape(-1,1))
    # concatenate along last axis
    arr = np.concatenate(data_cols, axis=1)
    return arr

arr = build_feature_array(df, FEATURE_COLUMNS)
print("Feature array shape (rows,features):", arr.shape)

# Window builder: X shape (N_windows, LOOKBACK, n_features) ; Y shape (N_windows, HORIZON)
def make_windows_from_array(arr, index, lookback=LOOKBACK, horizon=HORIZON):
    X, Y, idxs = [], [], []
    n_rows = arr.shape[0]
    # need at least lookback + horizon
    for i in range(lookback, n_rows - horizon + 1):
        X.append(arr[i-lookback:i, :])
        Y.append(arr[i:i+horizon, 0])   # target is first column (demand)
        idxs.append(index[i+horizon-1])
    if len(X) == 0:
        return np.empty((0,lookback,arr.shape[1])), np.empty((0,horizon)), []
    return np.array(X, dtype=float), np.array(Y, dtype=float), idxs

X_all, Y_all, IDX_all = make_windows_from_array(arr, df.index, LOOKBACK, HORIZON)
N = len(X_all)
print("Windows total:", N)

if N == 0:
    # print diagnostic suggestions
    print(">>> No windows created. Diagnostics:")
    print(" - len(df) =", len(df))
    print(" - LOOKBACK + HORIZON =", LOOKBACK + HORIZON)
    print("Possible fixes: reduce LOOKBACK/HORIZON or use a longer timeseries.")
    # still export globals so caller can inspect
    globals().update({
        'FEATURE_COLUMNS': FEATURE_COLUMNS,
        'X_all': X_all, 'Y_all': Y_all, 'IDX_all': IDX_all
    })
    raise RuntimeError("No windows created. See diagnostics printed above.")

# train/val/test split (by windows)
n_train = int(0.7 * N)
n_val   = int(0.15 * N)
n_test  = N - n_train - n_val
print(f"Train/Val/Test windows: {n_train} / {n_val} / {n_test}")

X_tr, Y_tr = X_all[:n_train], Y_all[:n_train]
X_val, Y_val = X_all[n_train:n_train+n_val], Y_all[n_train:n_train+n_val]
X_test, Y_test = X_all[n_train+n_val:], Y_all[n_train+n_val:]
idx_test = IDX_all[n_train+n_val:]

print("X_tr shape:", X_tr.shape, "Y_tr shape:", Y_tr.shape)
print("X_test shape:", X_test.shape, "Y_test shape:", Y_test.shape)

# compute index of 'solar' (generation proxy) in FEATURE_COLUMNS if present
if 'solar' in FEATURE_COLUMNS:
    gen_proxy_idx = FEATURE_COLUMNS.index('solar')
elif 'wind_gen' in FEATURE_COLUMNS:
    gen_proxy_idx = FEATURE_COLUMNS.index('wind_gen')
else:
    gen_proxy_idx = None

# create gen_proxy for economic metrics (use last time-step feature value in input window)
if gen_proxy_idx is not None:
    gen_proxy = X_test[:, -1, gen_proxy_idx]
else:
    gen_proxy = np.zeros(X_test.shape[0])

# expose useful names to globals so downstream cells don't break
globals().update({
    'FEATURE_COLUMNS': FEATURE_COLUMNS,
    'X_all': X_all, 'Y_all': Y_all, 'IDX_all': IDX_all,
    'X_tr': X_tr, 'Y_tr': Y_tr, 'X_val': X_val, 'Y_val': Y_val,
    'X_test': X_test, 'Y_test': Y_test, 'idx_test': idx_test,
    'gen_proxy_idx': gen_proxy_idx, 'gen_proxy': gen_proxy,
    'LOOKBACK': LOOKBACK, 'HORIZON': HORIZON
})

# quick check: ensure downstream code expecting X_tr shape will work
print("Done. Exposed variables: X_tr, Y_tr, X_val, Y_val, X_test, Y_test, idx_test, FEATURE_COLUMNS")


Feature columns used (in order): ['demand', 'solar', 'hour', 'dayofyear', 'weekday', 'Time_step', 'Year', 'Month', 'Day', 'Hour', 'AL', 'AT', 'BE', 'BG', 'BA', 'CH', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI', 'FR', 'HR', 'HU', 'XK', 'IE', 'IS', 'IT', 'LT', 'LU', 'LV', 'ME', 'MK', 'NL', 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK', 'UK']
Feature array shape (rows,features): (262945, 46)
Windows total: 262874
Train/Val/Test windows: 184011 / 39431 / 39432
X_tr shape: (184011, 48, 46) Y_tr shape: (184011, 24)
X_test shape: (39432, 48, 46) Y_test shape: (39432, 24)
Done. Exposed variables: X_tr, Y_tr, X_val, Y_val, X_test, Y_test, idx_test, FEATURE_COLUMNS


In [10]:
# Cell 7: diagnostics to ensure there is enough data
print("Rows:", len(df))
print("Index range:", df.index.min(), "->", df.index.max())
print("Inferred freq:", pd.infer_freq(df.index))
print("NaNs per column:\n", df.isna().sum())
if len(df) < (LOOKBACK + HORIZON + 1):
    print(f"WARNING: dataset too short for LOOKBACK={LOOKBACK}, HORIZON={HORIZON}. Consider reducing these or using more data.")


Rows: 262945
Index range: 1986-01-01 00:00:00 -> 2015-12-31 00:00:00
Inferred freq: h
NaNs per column:
 Time_step    0
Year         0
Month        0
Day          0
Hour         0
AL           0
AT           0
BE           0
BG           0
BA           0
CH           0
CY           0
CZ           0
DE           0
DK           0
EE           0
EL           0
ES           0
FI           0
FR           0
HR           0
HU           0
XK           0
IE           0
IS           0
IT           0
LT           0
LU           0
LV           0
ME           0
MK           0
NL           0
NO           0
PL           0
PT           0
RO           0
RS           0
SE           0
SI           0
SK           0
UK           0
hour         0
dayofyear    0
weekday      0
demand       0
solar        0
dtype: int64


In [12]:
# Cell 9
from sklearn.metrics import mean_absolute_error, mean_squared_error
def MAE_(y,yhat): return float(mean_absolute_error(y,yhat))
def RMSE(y,yhat): return float(np.sqrt(mean_squared_error(y,yhat)))
def R2_(y,yhat): return float(r2_score(y,yhat))
def pinball(y,yhat,q=0.5):
    e = y - yhat
    return float(np.mean(np.maximum(q*e, (q-1)*e)))
def compute_pes(gen, dem): return float(np.sum(np.maximum(0, gen - dem)))
def compute_pec(imported, tariff=0.1): return float(np.sum(imported) * tariff)
def compute_gr(sold, price=0.08): return float(np.sum(sold) * price)


In [ ]:
# Cell 10: rolling ARIMA last-step forecast
demand_series = df['demand']
def rolling_arima(series, train_size, horizon=HORIZON, order=(5,1,0)):
    preds = []; idxs = []
    for start in range(train_size, len(series)-horizon+1, horizon):
        train = series.iloc[:start]
        try:
            m = ARIMA(train, order=order).fit()
            fc = m.forecast(steps=horizon)
        except Exception:
            fc = pd.Series(np.repeat(train.mean(), horizon), index=series.index[start:start+horizon])
        preds.extend(fc)
        idxs.extend(series.index[start:start+horizon])
    return pd.Series(preds, index=idxs)

arima_series = rolling_arima(demand_series, train_size=int(len(df)*0.7))
arima_last = arima_series.reindex(idx_test).fillna(method='ffill').values
print("ARIMA done, aligned length:", len(arima_last))


C:\Users\Admin\MINICONDA\envs\ml_env\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
